In [13]:
# Cell 1 — Import Required Libraries

import json
import ollama
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [14]:
# Cell 2 — Load and Validate Cleaned Data

df = pd.read_csv(
    "../data/processed/customer_support_tickets_clean.csv"
)

print(f"Duplicated tickets: {df.duplicated().sum()}")

invalid_type = (
    df["Ticket Type"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    f"Invalid or empty Ticket Type values: {invalid_type}"
)

invalid_description = (
    df["Ticket Description"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    f"Invalid or empty Ticket Description values: "
    f"{invalid_description}"
)

Duplicated tickets: 0
Invalid or empty Ticket Type values: 0
Invalid or empty Ticket Description values: 0


In [15]:
# Cell 3 — Recreate the ML Train/Test Pipeline

X = df["Ticket Description"]
y = df["Ticket Type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)

model = LogisticRegression()

model.fit(
    X_train_tfidf,
    y_train
)

print(f"Training tickets: {len(X_train)}")
print(f"Test tickets: {len(X_test)}")
print(f"TF-IDF features: {X_train_tfidf.shape[1]}")

Training tickets: 6775
Test tickets: 1694
TF-IDF features: 6290


In [16]:
# Cell 4 — Recreate the Controlled 25-Ticket Sample

def generate_prompt(ticket_description):
    prompt = f"""
Analyze the following customer-support ticket.

Ticket description:
{ticket_description}

Classify the ticket using exactly one of these categories:
- Technical Issue
- Refund Request
- Cancellation Request
- Billing Inquiry
- Product Inquiry

Return the analysis as JSON using exactly these four fields:

{{
    "Summary": "A short description of the customer's problem",
    "Category": "One of the five allowed ticket categories",
    "Sentiment": "The customer's expressed sentiment",
    "Urgency": "The estimated urgency of the problem"
}}

Return only the JSON object and no additional text.
"""
    return prompt


def llm_analysis(ticket_description):
    prompt = generate_prompt(ticket_description)

    response = ollama.chat(
        model="llama3.1",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    response_text = response["message"]["content"]
    analysis = json.loads(response_text)

    return analysis


ticket_categories = [
    "Technical Issue",
    "Refund Request",
    "Cancellation Request",
    "Billing Inquiry",
    "Product Inquiry"
]


ticket_sample = pd.concat([
    df[df["Ticket Type"] == ticket_type].sample(
        5,
        random_state=42
    )
    for ticket_type in ticket_categories
]).reset_index(drop=True)


display(
    ticket_sample[
        ["Ticket Description", "Ticket Type"]
    ]
)

print("\nTickets per category:")
print(
    ticket_sample["Ticket Type"].value_counts()
)

,Ticket Description,Ticket Type
0,I'm having an issue with the {product_purchase...,Technical Issue
1,I'm having an issue with the {product_purchase...,Technical Issue
2,I'm having an issue with the {product_purchase...,Technical Issue
3,I've forgotten my password for my {product_pur...,Technical Issue
4,I'm having an issue with the {product_purchase...,Technical Issue
5,I'm facing issues logging into my {product_pur...,Refund Request
6,I'm having an issue with the {product_purchase...,Refund Request
7,I'm having an issue with the {product_purchase...,Refund Request
8,I'm unable to access my {product_purchased} ac...,Refund Request
9,I'm having an issue with the {product_purchase...,Refund Request



Tickets per category:
Ticket Type
Technical Issue         5
Refund Request          5
Cancellation Request    5
Billing Inquiry         5
Product Inquiry         5
Name: count, dtype: int64


In [17]:
# Cell 5 — Generate ML Predictions

sample_tfidf = vectorizer.transform(
    ticket_sample["Ticket Description"]
)

ml_predictions = model.predict(
    sample_tfidf
)

print(f"ML predictions generated: {len(ml_predictions)}")
print(ml_predictions)

ML predictions generated: 25
['Technical Issue' 'Technical Issue' 'Technical Issue' 'Technical Issue'
 'Product Inquiry' 'Billing Inquiry' 'Product Inquiry' 'Refund Request'
 'Product Inquiry' 'Refund Request' 'Billing Inquiry'
 'Cancellation Request' 'Billing Inquiry' 'Cancellation Request'
 'Technical Issue' 'Billing Inquiry' 'Billing Inquiry' 'Product Inquiry'
 'Billing Inquiry' 'Billing Inquiry' 'Product Inquiry'
 'Cancellation Request' 'Billing Inquiry' 'Product Inquiry'
 'Technical Issue']


In [18]:
# Cell 6 — Generate LLM Predictions

llm_responses = [
    llm_analysis(description)
    for description in ticket_sample["Ticket Description"]
]

llm_predictions = [
    response["Category"]
    for response in llm_responses
]

print(f"LLM predictions generated: {len(llm_predictions)}")
print(llm_predictions)

LLM predictions generated: 25
['Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Product Inquiry', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Product Inquiry', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue', 'Technical Issue']


In [19]:
# Cell 7 — Build the Final Comparison DataFrame

comparison = pd.DataFrame({
    "Ticket Description":
        ticket_sample["Ticket Description"],

    "Actual Type":
        ticket_sample["Ticket Type"],

    "ML Prediction":
        ml_predictions,

    "LLM Prediction":
        llm_predictions
})


comparison["ML Correct"] = (
    comparison["Actual Type"]
    == comparison["ML Prediction"]
)


comparison["LLM Correct"] = (
    comparison["Actual Type"]
    == comparison["LLM Prediction"]
)


display(comparison)

,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct
0,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
1,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
2,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True
3,I've forgotten my password for my {product_pur...,Technical Issue,Technical Issue,Technical Issue,True,True
4,I'm having an issue with the {product_purchase...,Technical Issue,Product Inquiry,Product Inquiry,False,False
5,I'm facing issues logging into my {product_pur...,Refund Request,Billing Inquiry,Technical Issue,False,False
6,I'm having an issue with the {product_purchase...,Refund Request,Product Inquiry,Technical Issue,False,False
7,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False
8,I'm unable to access my {product_purchased} ac...,Refund Request,Product Inquiry,Technical Issue,False,False
9,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False


In [20]:
# Cell 8 — Create Detailed Error Categories

def categorize_result(row):
    if row["ML Correct"] and row["LLM Correct"]:
        return "Both Correct"

    elif row["ML Correct"] and not row["LLM Correct"]:
        return "ML Only Correct"

    elif not row["ML Correct"] and row["LLM Correct"]:
        return "LLM Only Correct"

    else:
        return "Both Wrong"


comparison["Error Category"] = comparison.apply(
    categorize_result,
    axis=1
)


print("Error Category Counts:")
print(
    comparison["Error Category"].value_counts()
)

display(comparison)

Error Category Counts:
Error Category
Both Wrong         11
ML Only Correct    10
Both Correct        4
Name: count, dtype: int64


,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct,Error Category
0,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True,Both Correct
1,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True,Both Correct
2,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Technical Issue,True,True,Both Correct
3,I've forgotten my password for my {product_pur...,Technical Issue,Technical Issue,Technical Issue,True,True,Both Correct
4,I'm having an issue with the {product_purchase...,Technical Issue,Product Inquiry,Product Inquiry,False,False,Both Wrong
5,I'm facing issues logging into my {product_pur...,Refund Request,Billing Inquiry,Technical Issue,False,False,Both Wrong
6,I'm having an issue with the {product_purchase...,Refund Request,Product Inquiry,Technical Issue,False,False,Both Wrong
7,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False,ML Only Correct
8,I'm unable to access my {product_purchased} ac...,Refund Request,Product Inquiry,Technical Issue,False,False,Both Wrong
9,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False,ML Only Correct


In [21]:
# Cell 9 — Analyze ML Errors by Actual Ticket Type

ml_error_summary = (
    comparison
    .groupby("Actual Type")
    .agg(
        Total_Tickets=("Actual Type", "size"),
        ML_Matches=("ML Correct", "sum")
    )
    .reset_index()
)


ml_error_summary["ML Errors"] = (
    ml_error_summary["Total_Tickets"]
    - ml_error_summary["ML_Matches"]
)


ml_error_summary["ML Match Rate"] = (
    ml_error_summary["ML_Matches"]
    / ml_error_summary["Total_Tickets"]
)


display(
    ml_error_summary.style.format({
        "ML Match Rate": "{:.2%}"
    })
)

,Actual Type,Total_Tickets,ML_Matches,ML Errors,ML Match Rate
0,Billing Inquiry,5,4,1,80.00%
1,Cancellation Request,5,2,3,40.00%
2,Product Inquiry,5,2,3,40.00%
3,Refund Request,5,2,3,40.00%
4,Technical Issue,5,4,1,80.00%


In [22]:
# Cell 10 — Analyze LLM Errors by Actual Ticket Type

llm_error_summary = (
    comparison
    .groupby("Actual Type")
    .agg(
        Total_Tickets=("Actual Type", "size"),
        LLM_Matches=("LLM Correct", "sum")
    )
    .reset_index()
)


llm_error_summary["LLM Errors"] = (
    llm_error_summary["Total_Tickets"]
    - llm_error_summary["LLM_Matches"]
)


llm_error_summary["LLM Match Rate"] = (
    llm_error_summary["LLM_Matches"]
    / llm_error_summary["Total_Tickets"]
)


display(
    llm_error_summary.style.format({
        "LLM Match Rate": "{:.2%}"
    })
)

,Actual Type,Total_Tickets,LLM_Matches,LLM Errors,LLM Match Rate
0,Billing Inquiry,5,0,5,0.00%
1,Cancellation Request,5,0,5,0.00%
2,Product Inquiry,5,0,5,0.00%
3,Refund Request,5,0,5,0.00%
4,Technical Issue,5,4,1,80.00%


In [23]:
# Cell 11 — Analyze ML vs. LLM Disagreements

disagreements = comparison[
    comparison["ML Prediction"]
    != comparison["LLM Prediction"]
]


print(
    f"ML and LLM disagreed on "
    f"{len(disagreements)} of "
    f"{len(comparison)} tickets."
)


display(
    disagreements[
        [
            "Ticket Description",
            "Actual Type",
            "ML Prediction",
            "LLM Prediction",
            "ML Correct",
            "LLM Correct",
            "Error Category"
        ]
    ]
)

ML and LLM disagreed on 18 of 25 tickets.


,Ticket Description,Actual Type,ML Prediction,LLM Prediction,ML Correct,LLM Correct,Error Category
5,I'm facing issues logging into my {product_pur...,Refund Request,Billing Inquiry,Technical Issue,False,False,Both Wrong
6,I'm having an issue with the {product_purchase...,Refund Request,Product Inquiry,Technical Issue,False,False,Both Wrong
7,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False,ML Only Correct
8,I'm unable to access my {product_purchased} ac...,Refund Request,Product Inquiry,Technical Issue,False,False,Both Wrong
9,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,True,False,ML Only Correct
10,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue,False,False,Both Wrong
11,I'm having an issue with the {product_purchase...,Cancellation Request,Cancellation Request,Technical Issue,True,False,ML Only Correct
12,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue,False,False,Both Wrong
13,I'm having an issue with the {product_purchase...,Cancellation Request,Cancellation Request,Technical Issue,True,False,ML Only Correct
15,I've noticed a software bug in the {product_pu...,Billing Inquiry,Billing Inquiry,Technical Issue,True,False,ML Only Correct


In [24]:
# Cell 12 — Inspect Both-Wrong Cases

both_wrong = comparison[
    comparison["Error Category"] == "Both Wrong"
]


print(
    f"Both models were wrong on "
    f"{len(both_wrong)} tickets."
)


display(
    both_wrong[
        [
            "Ticket Description",
            "Actual Type",
            "ML Prediction",
            "LLM Prediction"
        ]
    ]
)

Both models were wrong on 11 tickets.


,Ticket Description,Actual Type,ML Prediction,LLM Prediction
4,I'm having an issue with the {product_purchase...,Technical Issue,Product Inquiry,Product Inquiry
5,I'm facing issues logging into my {product_pur...,Refund Request,Billing Inquiry,Technical Issue
6,I'm having an issue with the {product_purchase...,Refund Request,Product Inquiry,Technical Issue
8,I'm unable to access my {product_purchased} ac...,Refund Request,Product Inquiry,Technical Issue
10,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue
12,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue
14,"I've recently set up my {product_purchased}, b...",Cancellation Request,Technical Issue,Technical Issue
17,I'm having an issue with the {product_purchase...,Billing Inquiry,Product Inquiry,Technical Issue
21,I'm having an issue with the {product_purchase...,Product Inquiry,Cancellation Request,Technical Issue
22,I'm having an issue with the {product_purchase...,Product Inquiry,Billing Inquiry,Technical Issue


In [25]:
# Cell 13 — Inspect Ticket Text and Dataset-Label Ambiguity

error_cases = comparison[
    comparison["Error Category"] != "Both Correct"
].copy()


error_cases["Contains Placeholder"] = (
    error_cases["Ticket Description"]
    .str.contains(
        r"\{[^}]+\}",
        regex=True,
        na=False
    )
)


error_cases["Description Length"] = (
    error_cases["Ticket Description"]
    .str.len()
)


display(
    error_cases[
        [
            "Ticket Description",
            "Actual Type",
            "ML Prediction",
            "LLM Prediction",
            "Error Category",
            "Contains Placeholder",
            "Description Length"
        ]
    ]
)


print(
    "Error cases containing placeholders:",
    error_cases["Contains Placeholder"].sum()
)


print(
    "Average description length among error cases:",
    error_cases["Description Length"].mean()
)

,Ticket Description,Actual Type,ML Prediction,LLM Prediction,Error Category,Contains Placeholder,Description Length
4,I'm having an issue with the {product_purchase...,Technical Issue,Product Inquiry,Product Inquiry,Both Wrong,True,303
5,I'm facing issues logging into my {product_pur...,Refund Request,Billing Inquiry,Technical Issue,Both Wrong,True,327
6,I'm having an issue with the {product_purchase...,Refund Request,Product Inquiry,Technical Issue,Both Wrong,True,296
7,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,ML Only Correct,True,276
8,I'm unable to access my {product_purchased} ac...,Refund Request,Product Inquiry,Technical Issue,Both Wrong,True,323
9,I'm having an issue with the {product_purchase...,Refund Request,Refund Request,Technical Issue,ML Only Correct,True,244
10,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue,Both Wrong,True,254
11,I'm having an issue with the {product_purchase...,Cancellation Request,Cancellation Request,Technical Issue,ML Only Correct,True,154
12,I'm having an issue with the {product_purchase...,Cancellation Request,Billing Inquiry,Technical Issue,Both Wrong,True,319
13,I'm having an issue with the {product_purchase...,Cancellation Request,Cancellation Request,Technical Issue,ML Only Correct,True,319


Error cases containing placeholders: 21
Average description length among error cases: 287.0


In [26]:
# Cell 14 — Summarize Final Experimental Results

total_tickets = len(comparison)

ml_matches = comparison["ML Correct"].sum()
llm_matches = comparison["LLM Correct"].sum()

ml_match_rate = ml_matches / total_tickets
llm_match_rate = llm_matches / total_tickets


final_summary = pd.DataFrame({
    "Method": [
        "TF-IDF + Logistic Regression",
        "Llama 3.1"
    ],

    "Matches": [
        ml_matches,
        llm_matches
    ],

    "Total Tickets": [
        total_tickets,
        total_tickets
    ],

    "Match Rate": [
        ml_match_rate,
        llm_match_rate
    ]
})


display(
    final_summary.style.format({
        "Match Rate": "{:.2%}"
    })
)


error_counts = (
    comparison["Error Category"]
    .value_counts()
)


print("\nError Category Counts:")

print(
    f"Both Correct: "
    f"{error_counts.get('Both Correct', 0)}"
)

print(
    f"ML Only Correct: "
    f"{error_counts.get('ML Only Correct', 0)}"
)

print(
    f"LLM Only Correct: "
    f"{error_counts.get('LLM Only Correct', 0)}"
)

print(
    f"Both Wrong: "
    f"{error_counts.get('Both Wrong', 0)}"
)

print(
    f"ML/LLM Prediction Disagreements: "
    f"{len(disagreements)}"
)

,Method,Matches,Total Tickets,Match Rate
0,TF-IDF + Logistic Regression,14,25,56.00%
1,Llama 3.1,4,25,16.00%



Error Category Counts:
Both Correct: 4
ML Only Correct: 10
LLM Only Correct: 0
Both Wrong: 11
ML/LLM Prediction Disagreements: 18


## Day 32 Final Error Analysis and Conclusions

### Experiment Design

Day 32 extended the Day 31 comparison by examining why the TF-IDF + Logistic Regression model and Llama 3.1 succeeded or failed on the controlled sample.

The same 25 tickets were retained, with 5 examples from each Ticket Type. No new classifier was trained specifically for these tickets and the comparison sample was not changed. This preserved the same conditions used in the Day 31 experiment.

### Overall Experimental Results

The TF-IDF + Logistic Regression classifier matched 14 of 25 dataset labels, for a 56% match rate.

Llama 3.1 matched 4 of 25 dataset labels, for a 16% match rate.

The error-category analysis produced:

- Both Correct: 4
- ML Only Correct: 10
- LLM Only Correct: 0
- Both Wrong: 11

The two systems also produced different category predictions on 18 of the 25 tickets.

### Traditional ML Error Patterns

Logistic Regression achieved its highest match rates for Billing Inquiry and Technical Issue, matching 4 of 5 tickets in each category (80%).

It matched 2 of 5 tickets (40%) in each of the remaining categories:

- Cancellation Request
- Product Inquiry
- Refund Request

This indicates that the ML classifier did not perform consistently across categories even though it outperformed the LLM in overall dataset-label agreement on this sample.

### LLM Error Patterns

Llama 3.1 matched 4 of 5 Technical Issue tickets (80%), but matched none of the sampled tickets labeled as:

- Billing Inquiry
- Cancellation Request
- Product Inquiry
- Refund Request

The LLM frequently classified descriptions from these categories as Technical Issue. This suggests that the model often responded to the apparent problem described in the text rather than reproducing the category assigned by the dataset.

For example, descriptions involving login difficulties, software problems, account-access problems, or other malfunction-like language can reasonably appear technical even when their dataset label belongs to another category.

### Both-Wrong Cases

Both approaches failed to match the dataset label on 11 of the 25 tickets.

This is an important result because Logistic Regression and Llama 3.1 are very different approaches. When both fail on the same ticket, the error cannot automatically be attributed to one specific modeling technique.

The both-wrong examples included tickets from all five categories. Several descriptions used generic problem language or described issues that could plausibly suggest categories different from the dataset label.

### Ticket Text and Dataset Limitations

Among the 21 tickets where at least one approach failed to match the dataset label, all 21 contained template placeholders such as `{product_purchased}`. The average description length among these error cases was 287 characters.

The presence of placeholders is therefore an observable characteristic of the error cases. However, this experiment does not prove that placeholders caused the classification errors because the analysis did not compare error rates between otherwise equivalent tickets with and without placeholders.

Manual inspection also showed that many descriptions contain generic wording such as having an "issue" with a product. In some cases, the apparent intent of the description may not clearly correspond to the dataset's Ticket Type.

This creates a distinction between:

1. matching the dataset label, and
2. producing a category that appears semantically reasonable from the ticket description.

The experiment measured the first of these. A prediction counted as incorrect whenever it differed from the dataset label, even if another category could appear reasonable from the text alone.

### ML vs. LLM Interpretation

The experiment does not demonstrate that traditional machine learning is universally better than an LLM for customer-support classification.

Instead, it shows that under this project's specific conditions:

- Logistic Regression matched the dataset labels more frequently.
- Llama 3.1 strongly favored Technical Issue in this controlled sample.
- The systems disagreed on 18 of 25 tickets.
- Both systems failed on 11 tickets.
- Many error cases contained generic/template-based descriptions and placeholders.
- Some ticket descriptions may provide insufficient or ambiguous information for inferring the dataset label from text alone.

The small sample size of 25 tickets also means that these results should be treated as an exploratory comparison rather than a general performance benchmark.

### Final Conclusion

The ML-vs.-LLM experiment demonstrated that model evaluation requires more than comparing a single accuracy or match-rate number. Error analysis revealed substantial differences in how the two approaches interpreted the same ticket descriptions and also exposed possible limitations in the relationship between the ticket text and dataset labels.

For this dataset and controlled sample, TF-IDF + Logistic Regression provided stronger agreement with the existing Ticket Type labels than Llama 3.1. However, the high number of both-wrong cases, generic descriptions, template placeholders, and potentially ambiguous text suggests that improving the underlying data and examining the text-label relationship may be as important as changing the classification model.

Future work could investigate data quality, standardized ticket descriptions, clearer category definitions, larger evaluation samples, improved preprocessing, alternative classifiers, and more constrained LLM prompts before considering either approach suitable for production use.